In [1]:
!pip install peft

In [11]:
import os
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# --- Text Preprocessing Imports ---
import nltk
from nltk.corpus import stopwords
from bs4 import BeautifulSoup

# Download stopwords once
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# --- 1. Setup Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device}")

# --- 2. Define Preprocessing Function ---
def preprocess_text(text):
    text = str(text) 
    # 1. Removal of HTML
    text = BeautifulSoup(text, "html.parser").get_text()
    
    # 2. To Lower Case
    text = text.lower()
    
    # 3. Removal of URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 4. Removal of Twitter Handles (@user)
    text = re.sub(r'@\w+', '', text)
    
    # 5. Removal of Hashtag symbol (keeping text)
    text = re.sub(r'#', '', text) 
    
    # 6. Removal of Placeholders
    text = re.sub(r'\[.*?\]', '', text)

    # 7. Removal of Punctuation & Non-letter Characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 8. Removal of Stopwords
    text = ' '.join([word for word in text.split() if word not in stop_words])
    
    # 9. Clean extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("✅ Setup & Preprocessing Function Ready!")

🚀 Device: cuda
✅ Setup & Preprocessing Function Ready!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nabil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
# --- 3. Define Custom Dataset Class ---
class CrisisDataset(Dataset):
    def __init__(self, df, processor, data_path="."):
        self.df = df
        self.processor = processor
        self.data_path = data_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row['tweet_text']
        label = row['label']
        img_path = os.path.join(self.data_path, row['image'])
        
        try:
            # Load and convert image
            image = Image.open(img_path).convert("RGB")
            
            # CLIP Processor handles Image Resizing/Norm + Text Tokenization
            encoding = self.processor(
                text=text, 
                images=image, 
                return_tensors="pt", 
                padding="max_length", 
                truncation=True, 
                max_length=77
            )
            
            # Remove batch dimension added by processor
            return {
                'pixel_values': encoding['pixel_values'].squeeze(0),
                'input_ids': encoding['input_ids'].squeeze(0),
                'attention_mask': encoding['attention_mask'].squeeze(0),
                'label': torch.tensor(label, dtype=torch.long)
            }
        except Exception as e:
            # If an image fails, return the next one (simple error handling)
            return self.__getitem__((idx + 1) % len(self.df))

print("✅ Dataset Class Defined!")

✅ Dataset Class Defined!


In [14]:
import os
import pandas as pd
# --- 4. Load & Clean Data ---
# Define paths (Adjust these to match your folder structure)
train_path = os.path.join("data/CrisisMMD/crisismmd_datasplit_all/crisismmd_datasplit_all/task_informative_text_img_train.tsv")
test_path = os.path.join("data/CrisisMMD/crisismmd_datasplit_all/crisismmd_datasplit_all/task_informative_text_img_test.tsv")

# Load DataFrames
train_df = pd.read_csv(train_path, sep='\t')
test_df = pd.read_csv(test_path, sep='\t')

# Function to encode labels
def encode_label(row):
    if row['label_image'] == 'informative': return 1
    elif row['label_image'] == 'not_informative': return 0
    return None

# Apply Label Encoding
train_df['label'] = train_df.apply(encode_label, axis=1)
test_df['label'] = test_df.apply(encode_label, axis=1)

# Drop NaNs
train_df = train_df.dropna(subset=['label'])
test_df = test_df.dropna(subset=['label'])
train_df['label'] = train_df['label'].astype(int)
test_df['label'] = test_df['label'].astype(int)

# --- APPLY PREPROCESSING HERE ---
print("⏳ Applying Text Preprocessing (Steps 1-9)...")
tqdm.pandas() # Progress bar
train_df['tweet_text'] = train_df['tweet_text'].progress_apply(preprocess_text)
test_df['tweet_text'] = test_df['tweet_text'].progress_apply(preprocess_text)

# Initialize Processor
model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)

# Create Datasets & Loaders
batch_size = 32 # Keep small for GPU memory
train_dataset = CrisisDataset(train_df, processor)
test_dataset = CrisisDataset(test_df, processor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"✅ Data Ready! Train: {len(train_df)}, Test: {len(test_df)}")

⏳ Applying Text Preprocessing (Steps 1-9)...


100%|██████████| 2237/2237 [00:00<00:00, 13259.07it/s]


✅ Data Ready! Train: 13608, Test: 2237


In [15]:
import torch
import torch.nn.functional as F
from transformers import CLIPModel , CLIPProcessor
from tqdm import tqdm

print("--- 1. ZERO-SHOT PROMPT ENGINEERING ---")

# Define diverse prompts for the 'Informativeness' task
templates = [
    "A social media photo showing {} regarding a disaster.",
    "A tweet containing {} about an emergency.",
    "Visual evidence of {}.",
    "An image that is {} for humanitarian aid."
]
class_names = ["irrelevant or not informative information", "informative and useful crisis information"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n✓ Device: {device}")

# Initialize base model in eval mode
base_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True).to(device)
base_clip.eval()
model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
# Pre-compute text embeddings for the ensemble
with torch.no_grad():
    zeroshot_weights = []
    for cls_name in class_names:
        texts = [template.format(cls_name) for template in templates]
        inputs = processor(text=texts, padding=True, return_tensors="pt").to(device)
        class_embeddings = base_clip.get_text_features(**inputs)
        class_embeddings /= class_embeddings.norm(dim=-1, keepdim=True)
        class_embedding = class_embeddings.mean(dim=0)
        class_embedding /= class_embedding.norm()
        zeroshot_weights.append(class_embedding)
    zeroshot_weights = torch.stack(zeroshot_weights, dim=1).to(device) # Shape: [512, 2]

# Evaluate on Test Loader
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Zero-Shot Inference"):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['label'].to(device)
        
        image_features = base_clip.get_image_features(pixel_values)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        
        logits = (100.0 * image_features @ zeroshot_weights)
        preds = torch.argmax(logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

from sklearn.metrics import accuracy_score, classification_report
print(f"Zero-Shot Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(classification_report(all_labels, all_preds, target_names=["Not Informative", "Informative"]))

--- 1. ZERO-SHOT PROMPT ENGINEERING ---

✓ Device: cuda


Zero-Shot Inference:   0%|          | 0/70 [00:00<?, ?it/s]d:\Anaconda\envs\tweet_project\lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Zero-Shot Inference: 100%|██████████| 70/70 [00:45<00:00,  1.53it/s]

Zero-Shot Accuracy: 0.7617
                 precision    recall  f1-score   support

Not Informative       0.71      0.86      0.78      1086
    Informative       0.83      0.67      0.74      1151

       accuracy                           0.76      2237
      macro avg       0.77      0.76      0.76      2237
   weighted avg       0.77      0.76      0.76      2237



In [16]:
import torch.nn as nn
import torch.optim as optim

print("\n--- 2. ADAPTER-BASED FINE TUNING ---")

class CLIPAdapterLayer(nn.Module):
    def __init__(self, dim=512, bottleneck=128, alpha=0.5):
        super().__init__()
        self.fc1 = nn.Linear(dim, bottleneck)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(bottleneck, dim)
        self.alpha = alpha

    def forward(self, x):
        return self.alpha * self.fc2(self.relu(self.fc1(x))) + (1 - self.alpha) * x

class CLIPWithAdapters(nn.Module):
    def __init__(self, base_model, num_classes=2):
        super().__init__()
        self.clip = base_model
        # Freeze CLIP
        for param in self.clip.parameters():
            param.requires_grad = False
            
        self.vis_adapter = CLIPAdapterLayer()
        self.txt_adapter = CLIPAdapterLayer()
        self.classifier = nn.Linear(512 * 2, num_classes)

    def forward(self, input_ids, attention_mask, pixel_values):
        with torch.no_grad():
            v_feat = self.clip.get_image_features(pixel_values)
            t_feat = self.clip.get_text_features(input_ids, attention_mask)
            v_feat = v_feat / v_feat.norm(dim=-1, keepdim=True)
            t_feat = t_feat / t_feat.norm(dim=-1, keepdim=True)
            
        v_adapted = self.vis_adapter(v_feat)
        t_adapted = self.txt_adapter(t_feat)
        
        combined = torch.cat([v_adapted, t_adapted], dim=1)
        return self.classifier(combined)

adapter_model = CLIPWithAdapters(base_clip).to(device)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, adapter_model.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Quick Training Loop (3 Epochs)
adapter_model.train()
for epoch in range(3):
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Adapter Epoch {epoch+1}"):
        optimizer.zero_grad()
        out = adapter_model(batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['pixel_values'].to(device))
        loss = criterion(out, batch['label'].to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Loss: {total_loss/len(train_loader):.4f}")


--- 2. ADAPTER-BASED FINE TUNING ---


Adapter Epoch 1:  94%|█████████▍| 401/426 [05:10<00:18,  1.37it/s]d:\Anaconda\envs\tweet_project\lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Adapter Epoch 1: 100%|██████████| 426/426 [05:28<00:00,  1.30it/s]


Loss: 0.3279


Adapter Epoch 2: 100%|██████████| 426/426 [05:52<00:00,  1.21it/s]


Loss: 0.2676


Adapter Epoch 3: 100%|██████████| 426/426 [05:32<00:00,  1.28it/s]

Loss: 0.2401


In [17]:
+from peft import LoraConfig, get_peft_model
from transformers import CLIPModel

print("\n--- 3. LORA FINE-TUNING ---")

# Load a fresh model
lora_base_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True).to(device)

# Configure LoRA to target Query and Value projection layers in the Attention blocks
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none"
)

# Wrap model with LoRA
peft_model = get_peft_model(lora_base_model, lora_config)
peft_model.print_trainable_parameters()

# Build a simple classifier head on top of the LoRA model
class LoRACrisisClassifier(nn.Module):
    def __init__(self, peft_clip):
        super().__init__()
        self.clip = peft_clip
        self.classifier = nn.Linear(1024, 2) # 512 img + 512 text

    def forward(self, input_ids, attention_mask, pixel_values):
        v_feat = self.clip.get_image_features(pixel_values)
        t_feat = self.clip.get_text_features(input_ids, attention_mask)
        combined = torch.cat([v_feat, t_feat], dim=1)
        return self.classifier(combined)

lora_classifier = LoRACrisisClassifier(peft_model).to(device)

# Optimizer targets the LoRA weights + Classifier head
optimizer_lora = optim.AdamW(filter(lambda p: p.requires_grad, lora_classifier.parameters()), lr=2e-4)

# Quick Training Loop (3 Epochs)
lora_classifier.train()
for epoch in range(3):
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"LoRA Epoch {epoch+1}"):
        optimizer_lora.zero_grad()
        out = lora_classifier(batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['pixel_values'].to(device))
        loss = criterion(out, batch['label'].to(device))
        loss.backward()
        optimizer_lora.step()
        total_loss += loss.item()
    print(f"Loss: {total_loss/len(train_loader):.4f}")


--- 3. LORA FINE-TUNING ---
trainable params: 983,040 || all params: 152,260,353 || trainable%: 0.6456


LoRA Epoch 1: 100%|██████████| 426/426 [37:17<00:00,  5.25s/it]


Loss: 0.3239


LoRA Epoch 2: 100%|██████████| 426/426 [37:08<00:00,  5.23s/it]


Loss: 0.2057


LoRA Epoch 3: 100%|██████████| 426/426 [37:07<00:00,  5.23s/it]

Loss: 0.1287


In [18]:
# Save the learned adapter weights
peft_model.save_pretrained("lora_crisis_clip_adapter")
print("✅ LoRA adapters saved to /lora_crisis_clip_adapter")

✅ LoRA adapters saved to /lora_crisis_clip_adapter


In [19]:
lora_classifier.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating LoRA"):
        out = lora_classifier(
            batch['input_ids'].to(device), 
            batch['attention_mask'].to(device), 
            batch['pixel_values'].to(device)
        )
        preds = torch.argmax(out, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['label'].cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=["Not Informative", "Informative"]))

Evaluating LoRA: 100%|██████████| 70/70 [01:57<00:00,  1.68s/it]

                 precision    recall  f1-score   support

Not Informative       0.89      0.86      0.88      1086
    Informative       0.88      0.90      0.89      1151

       accuracy                           0.88      2237
      macro avg       0.88      0.88      0.88      2237
   weighted avg       0.88      0.88      0.88      2237



In [20]:
import torchvision.transforms as T

# Define a simple TTA transform
tta_transforms = T.Compose([
    T.RandomHorizontalFlip(p=1.0),
    T.ColorJitter(brightness=0.1),
])

def predict_tta(model, batch, num_tta=3):
    # original prediction
    pixel_values = batch['pixel_values'].to(device)
    
    # Run original + augmented
    all_logits = [model(batch['input_ids'].to(device), batch['attention_mask'].to(device), pixel_values)]
    
    for _ in range(num_tta):
        aug_pixels = tta_transforms(pixel_values)
        all_logits.append(model(batch['input_ids'].to(device), batch['attention_mask'].to(device), aug_pixels))
        
    # Average the logits
    avg_logits = torch.stack(all_logits).mean(dim=0)
    return torch.argmax(avg_logits, dim=1)

In [24]:
class MultimodalDisasterClassifier(nn.Module):
    def __init__(self, model_id, num_classes=2):
        super(MultimodalDisasterClassifier, self).__init__()
        
        print(f"Loading {model_id} with SafeTensors...")
        self.clip = CLIPModel.from_pretrained(model_id, use_safetensors=True)
        
        # Increased capacity in the head (Standard 512 -> 256 -> 2)
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),   # Added BatchNorm for stability
            nn.ReLU(),
            nn.Dropout(0.3),       # Increased Dropout to 0.3
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        text_out = self.clip.get_text_features(input_ids=input_ids, attention_mask=attention_mask)
        img_out = self.clip.get_image_features(pixel_values=pixel_values)
        
        # Normalize
        text_out = text_out / text_out.norm(dim=-1, keepdim=True)
        img_out = img_out / img_out.norm(dim=-1, keepdim=True)
        
        combined = torch.cat((img_out, text_out), dim=1)
        logits = self.classifier(combined.float())
        return logits


In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

# --- A. RE-INITIALIZE YOUR BASE MODEL (CRITICAL) ---
# RESTART YOUR KERNEL BEFORE RUNNING THIS.
# Replace "your-model-id-here" with your actual model path (e.g., "openai/clip-vit-base-patch32")
MODEL_ID = "openai/clip-vit-base-patch32" # <-- UPDATE THIS ACCORDING TO YOUR THESIS CODE

model = MultimodalDisasterClassifier(model_id=MODEL_ID, num_classes=2) 
model.to("cuda" if torch.cuda.is_available() else "cpu")
device = next(model.parameters()).device

# --- B. DEFINE HELPERS ---
class MockConfig:
    def __init__(self):
        self.tie_word_embeddings = False
        self.use_return_dict = False
    def get(self, key, default=None):
        return getattr(self, key, default)

class EarlyStopper:
    def __init__(self, patience=4):
        self.patience, self.counter, self.best_acc, self.early_stop = patience, 0, 0.0, False
    def __call__(self, val_acc):
        if val_acc > self.best_acc:
            self.best_acc, self.counter = val_acc, 0
            return True
        self.counter += 1
        if self.counter >= self.patience: self.early_stop = True
        return False

# --- C. APPLY CLEAN LoRA ---
model.config = MockConfig()
lora_config = LoraConfig(
    r=64, 
    lora_alpha=128, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], 
    lora_dropout=0.1, 
    bias="none"
)

# 
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# --- D. TRAINING SETUP ---
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
early_stopper = EarlyStopper(patience=4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)


Loading openai/clip-vit-base-patch32 with SafeTensors...
trainable params: 5,898,240 || all params: 157,767,299 || trainable%: 3.7386


In [26]:
# 1. Find indices where model was WRONG on the validation/test set
model.eval()
wrong_indices = []

with torch.no_grad():
    for i, batch in enumerate(train_loader): # Checking the training set for "hard" samples
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['label'].to(device)
        
        logits = model(input_ids, attention_mask, pixel_values)
        preds = torch.argmax(logits, dim=1)
        
        # Identify indices where prediction != label
        mask = (preds != labels)
        batch_indices = [i*batch_size + j for j, x in enumerate(mask) if x]
        wrong_indices.extend(batch_indices)

# 2. Create the weights array
weights = torch.ones(len(train_df))

# 3. Increase weight for "Hard" samples (e.g., 5.0x multiplier)
# Ensure wrong_indices don't exceed the dataframe length
valid_wrong_indices = [idx for idx in wrong_indices if idx < len(train_df)]
weights[valid_wrong_indices] = 5.0 

print(f"Found {len(valid_wrong_indices)} hard samples to oversample.")

# 4. Create the sampler
sampler = torch.utils.data.WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

# 5. Re-initialize loader with sampler
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)

d:\Anaconda\envs\tweet_project\lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Found 6549 hard samples to oversample.
